In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../../DASH_learning/SoilData/SOIL_data_Final.csv')
df = df.dropna()
print(df.shape)

# Ensure WETCLASS is numeric
df['WETCLASS'] = pd.to_numeric(df['WETCLASS'], errors='coerce')

# Start by creating an empty classification column
df['ALC_SoilClass14'] = np.nan

# Define conditions and assignments in order (without stony soil and without TOTSTONE filters)
conditions = [
    # Peat
    (df['1_TEXTURE_CLASS'] == 'P') & (df['DEPTH_TOTAL'] >= 40),
    
    # Marine light silty soil
    df['ALC_SoilClass14'].isna() & (df['1_TEXTURE_CLASS'] == 'MZ'),
    
    # Shallow calcareous
    df['ALC_SoilClass14'].isna() & (df['DEPTH_TOTAL'] <= 50) & (df['1_CACO3'] == 'Y'),
    
    # Shallow non-calcareous
    df['ALC_SoilClass14'].isna() & (df['DEPTH_TOTAL'] <= 50) & (df['1_CACO3'] == 'N'),
    
    # Organic-mineral well drained
    df['ALC_SoilClass14'].isna() & (df['DEPTH_TOTAL'] > 50) & (df['1_TEXTURE_CLASS'].str.contains("org", case=False)) & (df['WETCLASS'] <= 2),
    
    # Organic-mineral moderate to poorly drained
    df['ALC_SoilClass14'].isna() & (df['DEPTH_TOTAL'] > 50) & (df['1_TEXTURE_CLASS'].str.contains("org", case=False)) & (df['WETCLASS'] >= 3),
    
    # Heavy clay soil
    df['ALC_SoilClass14'].isna() & (df['DEPTH_TOTAL'] > 50) & (df['1_TEXTURE_CLASS'].isin(['C', 'ZC', 'SC'])),
    
    # Light well-drained
    df['ALC_SoilClass14'].isna() & (df['DEPTH_TOTAL'] > 50) & (df['1_TEXTURE_CLASS'].isin(['S', 'LS', 'SL', 'SZL', 'ZL'])) & (df['WETCLASS'] <= 2),
    
    # Light moderate to poorly draining
    df['ALC_SoilClass14'].isna() & (df['DEPTH_TOTAL'] > 50) & (df['1_TEXTURE_CLASS'].isin(['S', 'LS', 'SL', 'SZL', 'ZL'])) & (df['WETCLASS'] >= 3),
    
    # Medium well-drained
    df['ALC_SoilClass14'].isna() & (df['DEPTH_TOTAL'] > 50) & (df['1_TEXTURE_CLASS'].isin(['SCL', 'CL', 'ZCL'])) & (df['WETCLASS'] <= 2),
    
    # Medium moderately draining
    df['ALC_SoilClass14'].isna() & (df['DEPTH_TOTAL'] > 50) & (df['1_TEXTURE_CLASS'].isin(['SCL', 'CL', 'ZCL'])) & (df['WETCLASS'] == 3),
    
    # Medium poorly draining
    df['ALC_SoilClass14'].isna() & (df['DEPTH_TOTAL'] > 50) & (df['1_TEXTURE_CLASS'].isin(['SCL', 'CL', 'ZCL'])) & (df['WETCLASS'] >= 4),
]

choices = [
    "Peat",
    "Marine light silty soil",
    "Shallow calcareous soil",
    "Shallow non calcareous soil",
    "Organic-mineral well drained soil",
    "Organic-mineral moderately to poorly drained soil",    
    "Heavy clay soil",
    "Light well-drained soil",
    "Light moderate to poorly draining soil",
    "Medium well-drained soil",
    "Medium moderately draining soil",
    "Medium poorly draining soil"
]

# Assign values using np.select
df['ALC_SoilClass14'] = np.select(conditions, choices, default='Unknown')

# Summarize
summary = df.groupby('ALC_SoilClass14').size().reset_index(name='n').sort_values(by='n', ascending=False)
dfSLU = pd.read_csv('../../../DASH_learning/SoilData/SOIL_CLASSIFICATION_LOOKUP.csv')

lookup_dict = dfSLU.set_index('ALC_SoilClass14')['Class_number'].to_dict()
df['target'] = df['ALC_SoilClass14'].map(lookup_dict)
print(df['target'].isna().sum())
df.to_csv('../../../DASH_learning/SoilData/SOIL_data_Final_class_12.csv', index=False)

# extract training samples using predictors and soil data

In [ ]:
import glob
import rasterio
import pandas as pd
import numpy as np
import os
import geopandas as gpd

# Path to the predictors
path = '../Data/predictData/RasterPredictors'
rasters = glob.glob(os.path.join(path, '*.tif'))

def extract_training_samples(shp_path, rasters):
    """
    Extract raster values at points from a shapefile.
    Returns a DataFrame with raster values + target + coordinates.
    """
    # Read shapefile
    gdf = gpd.read_file(shp_path)
    gdf = gdf.to_crs(epsg=27700)
    
    df = pd.DataFrame()
    
    # Loop through rasters
    for rfile in rasters:
        name = os.path.basename(rfile)[:-4]
        with rasterio.open(rfile) as src:
            coords = [(x, y) for x, y in zip(gdf.geometry.x, gdf.geometry.y)]
            values = list(src.sample(coords))
            values = np.array(values).flatten()
            df[name] = values
    
    # Add target and coordinates
    df["target"] = gdf["target"]
    df["EAST"] = gdf.geometry.x
    df["NORTH"] = gdf.geometry.y
    
    return df

# Example usage with two shapefiles
shp1 = '../../../DASH_learning/SoilData/soil_class_12.shp'
shp2 = '../../../DASH_learning/SoilData/Peat_Matthew.shp'

df1 = extract_training_samples(shp1, rasters)
df2 = extract_training_samples(shp2, rasters)

# Combine all training samples
df_all = pd.concat([df1, df2], ignore_index=True)

df_all.to_csv(os.path.join('../Data/trainingData/trainingSample.csv'), index=False)